# Prototype: atc

Reproduction check for `detectors/atc.py`. Not run by the pipeline: upload to Colab (File > Upload notebook).

**Runtime:** T4 GPU, Colab secret OPENROUTER_API_KEY.

**Decides:** Settle CodeLlama-7B fp16 vs nf4 for ATC on a 100+100 MBPP slice: AUROC vs Table 1 (7B, GPT-4o-mini: 91.23 at N=1, 93.44 at N=4, tolerance 3.5), seconds per sample, peak VRAM. Also a CRLF to LF run to see whether line endings inflate the paper's AUROC. Expect 1.5 to 3 h per variant: lower SLICE if the session is short.

Run top to bottom, then paste the printed comparison back into the chat.

**Session length:** set `SLICE = 20` and `VARIANTS = ["nf4"]` for a first pass; the full run can outlast a free Colab session.

In [ ]:
# %% [0] Setup. Runtime: T4 GPU. Colab secret OPENROUTER_API_KEY (for regenerating gpt-4o-mini code).
# Repo pins transformers==4.45.2 / torch==2.4.1; we keep Colab's torch+transformers and instead upcast
# logits to fp32 ourselves (4.45.2 did that inside LlamaForCausalLM.forward; >=4.46 returns fp16 logits,
# which makes the repo's probs*probs.log() NaN).
%pip install -q -U bitsandbytes
import os, re, json, time, gc, hashlib, resource, functools
import numpy as np, torch, torch.nn.functional as F, requests
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from datasets import load_dataset
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr

SLICE = 100          # MBPP test tasks: SLICE human + SLICE gpt-4o-mini samples
N_TASKS = 4          # paper headline uses N=4; N=1 is read off the first task
VARIANTS = ["fp16", "nf4"]
RUN_13B_CHECK = False  # optional: 13B-nf4 on the human slice vs the repo's shipped 13B scores
M7, M13 = "codellama/CodeLlama-7b-Instruct-hf", "codellama/CodeLlama-13b-Instruct-hf"
OUT = "/content/atc_proto"; os.makedirs(OUT, exist_ok=True)
print(torch.__version__, torch.cuda.get_device_name(0),
      round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2), "GiB")

In [ ]:
# %% [1] Paper's own human data: MBPP test split (task_ids 11..510), raw `code` field (99.8% CRLF).
mbpp = sorted(load_dataset("google-research-datasets/mbpp", "full", split="test"), key=lambda r: r["task_id"])[:SLICE]
human = {r["task_id"]: r["code"] for r in mbpp}

In [ ]:
# %% [2] Paper's LLM data is NOT released (email-gated). Regenerate gpt-4o-mini with the repo's exact prompt,
# sampling params and sanitizer (code_datasets/prompts.py, response_generator.py).
SYSTEM = """You serve as a programming assistant. I will first give you a programming challenge (Note that it is in Markdown format, and the math formula is inline latex).
You need to provide a {language} solution for the challenge."""
INSTR = """{problem}

Let's solve the problem step by step. You can first try to understand and analyze the problem. Then provide a {language} solution for the coding challenge above.
The {language} code should be organized in a single markdown block. Please do not add extra explanation for the code."""

def sanitize(response, model_name="gpt-4o-mini"):
    m = tuple(re.finditer(r"```(?:py(?:thon)?)?\n(.*?)```", response, re.DOTALL))
    response = m[0].group(1).strip() if len(m) == 1 else ""
    for prefix in ['# Test', '# Example']:
        response = response.split(prefix)[0]
    if 'gpt-4o' in model_name:
        bad = ['# Input and output', '# Input reading', '# Input Reading', '# Read input', '# Read Input',
               '# Reading input', '# Reading Input', '# Read the input', '# Read the Input']
        for p in bad:
            if p in response:
                s = response.split(p, maxsplit=1)
                if len(s[0]) > 0.5 * len(s[1]):
                    response = s[0].strip()
        response = '\n'.join(x for x in response.splitlines() if not any(x.strip().startswith(p) for p in bad))
        if response.endswith('\n'):
            response = response.rstrip('\n') + '\n'
    lines = [l.strip() for l in response.strip().split('\n') if l.strip()]
    if lines and lines[0].startswith("def ") and (len(lines) < 2 or all(l.startswith("#") or l == "pass" for l in lines[1:])):
        return ""
    return response

from google.colab import userdata
KEY = userdata.get("OPENROUTER_API_KEY")
gen_path = f"{OUT}/mbpp_gpt-4o-mini.jsonl"
llm = {int(k): v for l in open(gen_path) for k, v in json.loads(l).items()} if os.path.exists(gen_path) else {}
for r in mbpp:
    if r["task_id"] in llm:
        continue
    for _ in range(10):  # repo retries up to 100 until the sanitizer yields code
        resp = requests.post("https://openrouter.ai/api/v1/chat/completions",
            headers={"Authorization": f"Bearer {KEY}"}, timeout=120,
            json={"model": "openai/gpt-4o-mini", "temperature": 0.7, "top_p": 0.95, "max_tokens": 4096,
                  "messages": [{"role": "system", "content": SYSTEM.format(language="python")},
                               {"role": "user", "content": INSTR.format(problem=r["text"], language="python")}]}).json()
        code = sanitize(resp["choices"][0]["message"]["content"])
        if code:
            llm[r["task_id"]] = code
            open(gen_path, "a").write(json.dumps({r["task_id"]: code}) + "\n")
            break
ids = [t for t in human if t in llm]
print(len(ids), "paired tasks; served model:", resp.get("model") if "resp" in dir() else "cached")

In [ ]:
# %% [3] ATC, ported verbatim from detection/detector.py (EntropyDetector + compute_score_infer_task,
# prompt_style='regular', comments:0,docstrings:0). Only change: fp32 log_softmax entropy (see cell 0).
# Tasks are generated one at a time (batch 1, as the repo's per-seed reruns): num_return_sequences=4 would
# multiply the KV cache by 4 (~0.5 MiB/token at 7B fp16) and can OOM next to 12.55 GiB of fp16 weights.
def _chat_tokens(tok, chat):
    t_wo = tok.apply_chat_template(chat[:-1], tokenize=False)
    t = tok.apply_chat_template(chat, tokenize=False)
    n_wo = tok(t_wo, return_tensors="pt")["input_ids"].size(1)
    enc = tok(t, return_tensors="pt", return_offsets_mapping=True)   # repo quirk: double BOS, kept
    return enc, t, enc["input_ids"].size(1) - n_wo

def _ignore_tokens(enc, text, start):
    ranges = [(m.start() + start, m.end() + start) for m in re.finditer(r'^\s*(#.*)$', text[start:], flags=re.MULTILINE)]
    ranges += [(m.start(1) + start, m.end(1) + start) for m in re.finditer(r'\s*(""".*?""")', text[start:], flags=re.DOTALL)]
    offs = enc.pop("offset_mapping").squeeze().tolist()
    return [[i for i, (s0, e0) in enumerate(offs) if s0 >= s and e0 <= e] for s, e in ranges]

@torch.no_grad()
def neg_entropy(model, tok, chat):
    enc, text, n = _chat_tokens(tok, chat)
    groups = _ignore_tokens(enc, text, text.index('```python'))
    L = enc["input_ids"].size(1)
    logits = model(**{k: v.to(model.device) for k, v in enc.items()}).logits[0, -(n + 1):-1].float()
    logp = F.log_softmax(logits, -1)
    ent = -(logp.exp() * logp).sum(-1)                      # entropy indices L-1-n .. L-2
    mask = torch.ones(L - 1, device=ent.device)
    for g in groups:
        mask[[i for i in g if i != L - 1]] = 0               # repo quirk: token index used as entropy index
    mask = mask[-n:]
    return -((ent * mask).sum() / mask.sum()).item()

@torch.no_grad()
def infer_tasks(model, tok, code, n, seed):
    chat = [{"role": "system", "content": "You are a Python developer."},
            {"role": "user", "content": "Based on the provided code snippet, create a simple one line task description that, when given to an LLM, would likely result in the generation of a similar piece of code. The output should be wrapped in <task> and </task>. \n\n ```python\n" + code + "\n```"}]
    ids = tok(tok.apply_chat_template(chat, tokenize=False), add_special_tokens=False, return_tensors="pt").input_ids.to(model.device)
    tasks = []
    for i in range(n):
        torch.manual_seed(seed * 101 + i)
        out = model.generate(ids, attention_mask=ids.ne(tok.eos_token_id), max_length=ids.shape[-1] + 1000,
                             temperature=0.7, top_p=0.95, do_sample=True, pad_token_id=tok.eos_token_id)
        # repo quirk kept: decode includes the prompt, which itself contains "<task>"
        tasks.append(tok.decode(out[0], skip_special_tokens=True).split("<task>")[-1].split("</task>")[0].strip())
    return tasks

def atc(model, tok, code, n=N_TASKS, seed=0):
    """Returns (per-task scores, gen seconds, fwd seconds). Score = -mean code-token entropy; higher = LLM."""
    torch.cuda.synchronize(); t0 = time.perf_counter()
    tasks = infer_tasks(model, tok, code, n, seed)
    torch.cuda.synchronize(); t1 = time.perf_counter()
    s = [neg_entropy(model, tok, [
            {"role": "system", "content": "You are a Python developer."},
            {"role": "user", "content": t + '\n\n The python code should be organized in a single markdown block.'},
            {"role": "assistant", "content": "```python\n" + code + "\n```"}]) for t in tasks]
    torch.cuda.synchronize()
    return s, t1 - t0, time.perf_counter() - t1

def load(name, variant):
    q = dict(quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
             bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)) if variant == "nf4" else {}
    t = time.perf_counter()
    try:
        m = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float16, device_map="cuda", **q)
    except TypeError:
        m = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.float16, device_map="cuda", **q)
    return m.eval(), AutoTokenizer.from_pretrained(name), time.perf_counter() - t

In [ ]:
# %% [4] Run both variants on the same slice. Human scored raw (paper) and with CRLF->LF (our pipeline's likely form).
def tpr_at_10(h, l):
    t = np.quantile(h, 0.9, method="higher"); return float(np.mean(np.asarray(l) > t))

results = {}
for v in VARIANTS:
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    model, tok, load_s = load(M7, v)
    atc(model, tok, human[ids[0]], n=1)                                # warmup
    torch.cuda.reset_peak_memory_stats()
    R = {"load_s": load_s, "host_maxrss_gb": resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 2**20}
    arms = [("human", human), ("llm", llm)]
    if v == VARIANTS[0]:  # CRLF->LF arm once: does the paper's MBPP separation lean on line endings?
        arms.append(("human_lf", {k: c.replace("\r\n", "\n") for k, c in human.items()}))
    for arm, codes in arms:
        rows = [atc(model, tok, codes[k], seed=k) for k in ids]
        R[arm] = np.array([r[0] for r in rows])                       # [n_samples, N_TASKS]
        R[arm + "_t"] = np.array([[r[1], r[2]] for r in rows])
    R["peak_vram_gib"] = torch.cuda.max_memory_allocated() / 2**30
    # headroom stress: longer codes through task generation + scoring (MBPP is short; our Problems may not be)
    R["stress"] = {}
    for chars in (4000, 8000, 12000):
        long_code = "\n".join(llm[k] for k in ids)[:chars]
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
        try:
            atc(model, tok, long_code, n=1, seed=1); peak = round(torch.cuda.max_memory_allocated() / 2**30, 2)
        except torch.OutOfMemoryError:
            peak = "OOM"
        R["stress"][len(tok(long_code).input_ids)] = peak
    assert all(np.isfinite(R[a]).all() for a, _ in arms), "non-finite scores"
    results[v] = R
    np.savez(f"{OUT}/{v}.npz", **{k: x for k, x in R.items() if isinstance(x, np.ndarray)})
    del model; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# %% [5] Comparison. Paper targets (7B detector, MBPP, GPT4om column): AUROC N=1 91.23, N=4 93.44 (Table 1).
# Acceptance: within +-3.5 AUROC points (2.2 sd of 100+100 slices, measured on the repo's shipped scores).
TARGET = {1: 91.23, 4: 93.44}
for v, R in results.items():
    for n in (1, N_TASKS):
        for harm in [x for x in ("human", "human_lf") if x in R]:
            h, l = R[harm][:, :n].mean(1), R["llm"][:, :n].mean(1)
            a = 100 * roc_auc_score(np.r_[np.zeros(len(h)), np.ones(len(l))], np.r_[h, l])
            tgt = TARGET.get(n)
            print(f"{v:5s} N={n} {harm:8s} AUROC={a:6.2f} target={tgt} "
                  f"{'PASS' if tgt and abs(a - tgt) <= 3.5 else '----'} TPR@10%FPR={100*tpr_at_10(h, l):5.1f}")
    t = np.r_[R["human_t"], R["llm_t"]]
    print(f"{v:5s} s/sample (N={N_TASKS}): gen={t[:,0].mean():.2f} fwd={t[:,1].mean():.2f} total={t.sum(1).mean():.2f}"
          f" | load {R['load_s']:.0f}s host maxrss {R['host_maxrss_gb']:.1f} GiB | peak VRAM {R['peak_vram_gib']:.2f} GiB"
          f" | stress tokens->peak GiB {R['stress']}")
if len(results) == 2:
    a, b = results["fp16"], results["nf4"]
    for arm in ("human", "llm"):
        print(f"fp16 vs nf4 per-sample Spearman ({arm}, N={N_TASKS}):",
              round(spearmanr(a[arm].mean(1), b[arm].mean(1)).correlation, 3))
# Paper anchor for speed: 0.99 s task generation per MBPP sample, CodeLlama-7B, RTX6000 (§4.9).

In [ ]:
# %% [6] Optional implementation check against the only per-sample artifact: shipped 13B MBPP human scores.
if RUN_13B_CHECK:
    base = "https://raw.githubusercontent.com/maorash/ATC/main/results/codellama13b_seed_{}/scores_atc/entropy/comments_0_docstrings_0/google-research-datasets_mbpp_human_scores.jsonl"
    shipped = {s: {int(k): v["human_score"] for l in requests.get(base.format(s)).text.splitlines() for k, v in json.loads(l).items()} for s in (142, 242)}
    model, tok, _ = load(M13, "nf4")
    k13 = [k for k in ids if k in shipped[142]]
    ours = [atc(model, tok, human[k], n=1, seed=k)[0][0] for k in k13]
    print("13B-nf4 ours vs shipped seed142 Spearman:", round(spearmanr(ours, [shipped[142][k] for k in k13]).correlation, 3),
          "| ceiling seed142 vs seed242:", round(spearmanr([shipped[142][k] for k in k13], [shipped[242][k] for k in k13]).correlation, 3))
    del model; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# %% [C] detectors/atc.py once the prototype passes: cell 3's functions plus this module-level part.
VARIANT = "fp16"   # set from cell 5's result

@functools.cache
def _model():
    m, t, _ = load(M7, VARIANT)
    return m, t

def score(codes):
    m, t = _model()
    seeds = [int(hashlib.sha1(c.encode()).hexdigest()[:8], 16) for c in codes]   # same code -> same Score
    return [float(np.mean(atc(m, t, c, n=4, seed=s)[0])) for c, s in zip(codes, seeds)]

# Smoke test. No paper threshold: predict at 10% FPR, from cell 4's human Scores (N=4 mean) for VARIANT.
T = np.quantile(results[VARIANT]["human"].mean(1), 0.9, method="higher")
snippets, truth = [human[ids[0]], llm[ids[0]]], ["Human", "AI"]   # second should score higher
for code, p, t in zip(snippets, score(snippets), truth):
    print(f"score {p:.4f} (threshold {T:.4f}) -> predict {'AI' if p > T else 'Human'}, truth {t}")
    print("\n".join(code.splitlines()[:5]), "\n" + "-" * 40)